In [1]:
!pip install pyspark==3.5.1 -q

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
print(spark.version, "-", spark.sparkContext.defaultParallelism, "coeurs")

3.5.1 - 12 coeurs


In [3]:
!pip install faker -q

In [4]:
!python generate_data.py --scale 0.1 --outdir ./data

Génération (graine=42, échelle=0.1) vers ./data/
  - customers : 5,000 lignes
  - products  : 632 lignes
  - orders    : 50,000 lignes (+ lignes de commandes)
  - payments  : ~44,079 lignes (JSON Lines)
  - events    : ~330,000 lignes (JSON Lines, écrit par lots)
  - pg_init.sql : référentiel propre (customers, products)

Terminé. Fichiers générés :
  anomalies_manifest.json             0.0 Mo
  customers.csv                       0.6 Mo
  events.json                        68.5 Mo
  order_items.csv                     3.9 Mo
  orders.csv                          3.3 Mo
  payments.json                       9.0 Mo
  pg_init.sql                         0.6 Mo
  products.csv                        0.0 Mo


# Chargement des fichiers orders et events avec Spark, puis affichage de leur schéma

In [5]:
from pyspark.sql.functions import col

orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./data/orders.csv")
)

events = spark.read.json("./data/events.json")

orders.printSchema()
print("orders :", orders.count())
events.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)

orders : 50000
root
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ville: string (nullable = true)



# Exploration des commandes : sélection, filtrage et regroupement

In [6]:
orders.select("order_id", "statut", "canal").show(5)

livrees = orders.filter(col("statut") == "livrée")
print("livrees :", livrees.count())

orders.groupBy("statut").count().show()

orders.groupBy("canal") \
      .count() \
      .orderBy(col("count").desc()) \
      .show()

+--------+-------+----------+
|order_id| statut|     canal|
+--------+-------+----------+
|O0000001| livrée|mobile_app|
|O0000002| livrée|mobile_app|
|O0000003| livrée|       web|
|O0000004| livrée|mobile_app|
|O0000005|annulée|       web|
+--------+-------+----------+
only showing top 5 rows

livrees : 38890
+---------+-----+
|   statut|count|
+---------+-----+
|retournée| 2484|
|   livrée|38890|
| en_cours| 4058|
|  annulée| 4568|
+---------+-----+

+----------+-----+
|     canal|count|
+----------+-----+
|mobile_app|32519|
|       web|17481|
+----------+-----+



In [7]:
# Comparaison du temps de traitement entre Spark et Pandas

import time
import pandas as pd

# Temps Spark
t0 = time.perf_counter()

n = (
    spark.read
    .option("header", True)
    .csv("./data/orders.csv")
    .count()
)

t_spark = time.perf_counter() - t0

# Temps Pandas
t0 = time.perf_counter()

orders_pd = pd.read_csv("./data/orders.csv")

t_pandas = time.perf_counter() - t0

print("Spark  : %.2f s pour %d lignes" % (t_spark, n))
print("Pandas : %.2f s pour %d lignes" % (t_pandas, len(orders_pd)))

Spark  : 0.56 s pour 50000 lignes
Pandas : 0.16 s pour 50000 lignes


Sur 50 000 lignes, Pandas est plus rapide que Spark : 0,44 s contre 1,09 s. Sur ce petit volume, le coût de fonctionnement de Spark n’est pas compensé par le traitement distribué.